In [2]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

load_dotenv()

model = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [3]:
def create_outline(state: BlogState) -> BlogState:
    title = state['title']

    prompt = f'Generate a detailed outline for a blog on the topic: {title}'

    outline = model.invoke(prompt).text

    state['outline'] = outline
    return state

In [4]:
def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the following outline \n {outline}'

    content = model.invoke(prompt).text

    state['content'] = content
    return state

In [5]:
graph = StateGraph(BlogState)

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

initial_state = {'title': 'Rise of AI in India'}
final_state = workflow.invoke(initial_state)

print(final_state)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'title': 'Rise of AI in India', 'outline': 'Here is a comprehensive, highly detailed blog outline on the **"Rise of AI in India."** This structure is designed for a long-form, authoritative blog post (1,500–2,500 words) aimed at tech enthusiasts, business leaders, and policymakers.\n\n---\n\n# Blog Title Options:\n* **Option 1:** From IT Hub to AI Engine: The Rise of Artificial Intelligence in India\n* **Option 2:** India’s AI Renaissance: How the Nation is Shaping the Future of Tech\n* **Option 3:** The IndiaAI Revolution: Opportunities, Innovations, and the Road Ahead\n\n---\n\n## **I. Introduction**\n* **The Hook:** Start with a compelling statistic or scenario (e.g., India’s rapid jump in global AI readiness, or how an Indian farmer uses a voice-based AI app in their local dialect).\n* **Context:** India’s journey from being the world’s back office (IT services boom in the 2000s) to becoming a global frontrunner in AI innovation.\n* **Thesis Statement:** Driven by government visio